In [ ]:
import numpy as np 
import pandas as pd 
import os 
import sys 
sys.path.insert(1,'../../../../arimaclust/')
from arimaclust.models.centroid import ConsistentARMAClustering

In [ ]:
data_root = 'data/'
data_paths = sorted([os.path.join(data_root,path) for path in os.listdir(data_root)],reverse=True)


In [ ]:
def get_key(d,key):
    try: 
        return d[key]
    except Exception as e:
        return None 
    

def get_currency_series(df,symb):
    x = df['rates'].apply(lambda x: get_key(x,symb))
    return pd.Series(index=x.index,data=x.values,name=symb)

In [ ]:
df_= pd.read_json(data_paths[0])
currencies = df_['rates'].apply(lambda x: list(x.keys()))
unique_currencies = []
for row in currencies:
    unique_currencies.extend(row)
unique_currencies = list(set(unique_currencies))
currencies_data = get_currency_series(df_,symb=unique_currencies[0])

for currency in unique_currencies[1:]:
    curr_data_current = get_currency_series(df_,symb=currency)
    currencies_data = pd.concat([currencies_data,curr_data_current],axis=1)
    
    
final_data = currencies_data


for file_path in data_paths[1:]:
    print(file_path)
    df_= pd.read_json(file_path)
    currencies = df_['rates'].apply(lambda x: list(x.keys()))
    unique_currencies = []
    for row in currencies:
        unique_currencies.extend(row)
    unique_currencies = list(set(unique_currencies))
    currencies_data = get_currency_series(df_,symb=unique_currencies[0])
    
    for currency in unique_currencies[1:]:
        curr_data_current = get_currency_series(df_,symb=currency)
        currencies_data = pd.concat([currencies_data,curr_data_current],axis=1)
    final_data = pd.concat([currencies_data,final_data],axis=0)
    
    

In [ ]:
final_data.plot()

In [ ]:
final_data = final_data - final_data.mean()

In [ ]:
X = final_data.values

In [ ]:
model = ConsistentARMAClustering(k=10,max_orders=(2,2),tau=100)
model.fit(X)